# **Analisis Sentimen Review Aplikasi Shopee di Google Play Store Menggunakan Metode Deep Learning**

Notebook ini melakukan analisis sentimen terhadap **review pengguna aplikasi Shopee di Google Play Store**.
Data review dikumpulkan menggunakan library `google-play-scraper` yang menarik review publik dari Play Store
(bahasa Indonesia, negara Indonesia) beserta metadata-nya (bintang/`score`, jumlah like, versi aplikasi, dll).

Alur analisis yang dilakukan:

1. **Pengumpulan & pembersihan data** — load dataset, cek duplikat dan data kosong.
2. **Preprocessing teks** — konversi emoji, cleaning, casefolding, normalisasi kata tidak baku,
   penghapusan stopword, dan tokenisasi (pipeline NLP bahasa Indonesia dengan library Sastrawi & NLTK).
3. **Analisis frekuensi & wordcloud** — melihat kata-kata yang paling sering muncul (unigram/bigram/trigram).
4. **Pelabelan sentimen (lexicon-based)** — setiap review diberi label `positive` / `neutral` / `negative`
   berdasarkan skor lexicon Indonesia yang dikombinasikan dengan lexicon khusus konteks e-commerce.
5. **Klasifikasi sentimen** — 5 skema model:
   - Skema 1: **SVM + TF-IDF** (80/20)
   - Skema 2: **Random Forest + TF-IDF** (70/30)
   - Skema 3: **Logistic Regression + TF-IDF** (80/20)
   - Skema 4: **FastText Embedding + KNN** (80/20)
   - Skema 5: **SentenceTransformer (pre-trained) + Bi-LSTM** (80/20)

Setiap skema dilakukan tuning hyperparameter dan evaluasi (accuracy, classification report, confusion matrix),
lalu model terbaik disimpan ke folder `shopee/model/`.

## 1. Pengumpulan Data (Scraping Google Play Store)

Review diambil dari Google Play Store menggunakan library [`google-play-scraper`](https://pypi.org/project/google-play-scraper/).

- `app_id` adalah package ID aplikasi yang diambil dari URL Play Store (untuk Shopee: `com.shopee.id`).
- Review diambil per *batch* 200 baris menggunakan `Sort.NEWEST` (terbaru dahulu), dilanjutkan dengan
  `continuation_token` sampai data habis.
- Setiap `reviewId` dicek dengan `set` agar tidak ada review duplikat yang terambil akibat pergeseran posisi.

> **Catatan:** Dataset sudah pernah di-scrape dan tersimpan di `playstore_reviews.csv`, jadi sel ini bersifat
> opsional — cukup dijalankan ulang jika ingin memperbarui dataset.

In [ ]:
%pip install google-play-scraper pandas

import time
import pandas as pd
from google_play_scraper import reviews, Sort

app_id = "com.shopee.id"  # Package ID Shopee di Google Play Store

all_reviews = []
continuation_token = None
seen_ids = set()

while True:
    batch, next_token = reviews(
        app_id,
        lang="id",
        country="id",
        sort=Sort.NEWEST,
        count=200,
        continuation_token=continuation_token
    )

    if not batch:
        break

    for review in batch:
        review_id = review.get("reviewId")
        if review_id and review_id not in seen_ids:
            seen_ids.add(review_id)
            all_reviews.append(review)

    print(f"Total review terkumpul: {len(all_reviews)}")

    if next_token is None or next_token == continuation_token:
        break

    continuation_token = next_token
    time.sleep(1)

df = pd.DataFrame(all_reviews)
df.to_csv("playstore_reviews.csv", index=False, encoding="utf-8-sig")
print(f"Total review tersimpan: {len(df)}")

## 2. Loading Dataset

Dataset review Play Store dibaca dari `playstore_reviews.csv`. Kolom yang tersedia antara lain:

| Kolom | Keterangan |
|---|---|
| `reviewId` | ID unik review |
| `userName` | Nama pengguna |
| `content` | **Teks review (fitur utama yang dianalisis)** |
| `score` | Rating bintang 1–5 dari pengguna |
| `thumbsUpCount` | Jumlah like review |
| `at` | Waktu review dibuat |
| `appVersion` | Versi aplikasi saat review ditulis |

In [ ]:
import pandas as pd

df = pd.read_csv("playstore_reviews.csv")

jumlah_data, kolom = df.shape
print(f"Jumlah data: {jumlah_data}")
print(f"Jumlah kolom: {kolom}")

df.info()

### Cek Duplikat Data

Review yang sama bisa terambil dua kali karena posisi review tergeser saat scraping.
Duplikat dihapus agar data training tidak bocor (*data leakage*).

In [ ]:
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

df.drop_duplicates(inplace=True)
print(f"Jumlah data setelah dihapus duplikat: {df.shape[0]}")

### Cek Data Kosong

Baris dengan nilai kosong dihapus karena tidak bisa dipakai untuk preprocessing maupun pelabelan.

In [ ]:
print(f"Jumlah data kosong: {df.isna().sum().sum()}")
print(f"Jumlah data sebelum dihapus: {df.shape[0]}")
df.dropna(inplace=True)
print(f"Jumlah data setelah dihapus data kosong: {df.shape[0]}")

## 3. Preprocessing Teks

Review Play Store adalah teks informal: banyak emoji, singkatan, typo, dan huruf berulang
(contoh: "aplikasinya bagus bgttt 🔥🔥", "gk bisa checkout2"). Teks mentah seperti ini sulit diproses
model, sehingga dilakukan rangkaian normalisasi berikut:

| Tahap | Fungsi | Penjelasan |
|---|---|---|
| 0 | `convert_emoji` | Emoji dikonversi menjadi kata bermakna (mis. 🔥 → "menakjubkan") agar sinyal sentimen tidak hilang |
| 1 | `cleaningText` | Hapus URL, mention, angka, tanda baca, dan huruf berulang |
| 2 | `casefoldingText` | Samakan semua huruf menjadi huruf kecil |
| 3 | `fix_slangwords` | Normalisasi kata tidak baku / singkatan ke kata baku (kamus `kamus_kata_baku` + kamus custom e-commerce) |
| 4 | `filteringText` | Hapus stopword (kata umum tanpa bobot sentimen) dengan Sastrawi, kecuali kata negasi seperti "tidak" agar makna tetap terjaga |
| 5 | `stemmingText` | *(Opsional, tidak dijalankan)* Pengembalian ke kata dasar — lambat untuk data besar |
| 6 | `tokenizingText` | Pecah kalimat menjadi list token kata |

> **Catatan kata negasi:** kata seperti `tidak`, `jangan`, `ga`, `nggak` sengaja **tidak dihapus** oleh stopword remover.
> Tanpa kata negasi, kalimat "tidak bagus" akan sama dengan "bagus" dan label sentimen menjadi terbalik.

In [ ]:
import re
import string
from io import BytesIO

import pandas as pd
import requests
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    ArrayDictionary,
    StopWordRemover,
    StopWordRemoverFactory,
)
import emoji

# Pastikan resource NLTK tersedia
nltk.download('punkt')

# 1. DOWNLOAD & BUAT KAMUS SLANGWORDS (kata tidak baku -> kata baku)
url = "https://github.com/analysisdatasentiment/kamus_kata_baku/raw/main/kamuskatabaku.xlsx"
response = requests.get(url, timeout=30)
response.raise_for_status()

file_excel = BytesIO(response.content)
slang_data = pd.read_excel(file_excel, engine="openpyxl")

# Kamus tambahan khusus konteks review aplikasi / e-commerce
custom_slang_data = {
    'gk': 'tidak',
    'gak': 'tidak',
    'nggak': 'tidak',
    'enggak': 'tidak',
    'tdk': 'tidak',
    'blm': 'belum',
    'bsa': 'bisa',
    'bisa2': 'bisa',
    'yg': 'yang',
    'dgn': 'dengan',
    'dr': 'dari',
    'utk': 'untuk',
    'tp': 'tapi',
    'gw': 'saya',
    'sy': 'saya',
    'aku2': 'aku',
    'bgt': 'banget',
    'bgtt': 'banget',
    'bgttt': 'banget',
    'bgtttt': 'banget',
    'banget2': 'banget',
    'brg': 'barang',
    'kbrng': 'barang',
    'resi': 'resi',
    'ongkir': 'ongkos kirim',
    'ongkirnya': 'ongkos kirim',
    'seller': 'penjual',
    'cs': 'customer service',
    'error2': 'error',
    'errornya': 'error',
    'lemott': 'lemot',
    'lambatt': 'lambat',
    'rc': 'receh',
    'rekber': 'rekening bersama',
    'wa': 'whatsapp',
    'ig': 'instagram',
    'apk': 'aplikasi',
    'app': 'aplikasi',
    'aplikasiny': 'aplikasi',
    'ngeload': 'loading',
    'ngeleg': 'lemot',
    'fif': 'later paylater',
    'joss': 'bagus',
    'jiss': 'bagus'
}

kamus_tidak_baku_dict = dict(
    zip(slang_data["tidak_baku"], slang_data["kata_baku"])
)
kamus_tidak_baku_dict.update(custom_slang_data)

# 2. INISIALISASI SASTRAWI DI LUAR FUNGSI (agar cepat untuk data besar)
stopword_factory = StopWordRemoverFactory().get_stop_words()
words_to_keep = {'tidak', 'jangan', 'ga', 'gak', 'nggak', 'enggak', 'belum', 'kan', 'sih', 'dong', 'tuh', 'n'}
custom_stopword = [word for word in stopword_factory if word not in words_to_keep]
print(f"Jumlah stopword setelah dihapus kata negasi penting: {len(custom_stopword)}")

dictionary_stop = ArrayDictionary(custom_stopword)
stopword_remover = StopWordRemover(dictionary_stop)

# Inisialisasi Stemmer (opsional, lambat untuk data besar)
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

In [ ]:
def cleaningText(text):
    if not isinstance(text, str):
        return ""
    text = text.replace(".", " ")
    text = text.replace(",", " ")
    text = text.replace("\n", " ")
    # Normalisasi angka 2 sebagai pengganti kata ulang ("bagus2" -> "bagus-bagus")
    text = re.sub(r'\b([a-zA-Z]+)(?:2|["\u201c\u201d])', r'\1-\1', text)
    # Pisahkan kata yang dihubungkan tanda hubung
    text = re.sub(r'(?<=[a-zA-Z])-(?=[a-zA-Z])', ' ', text)
    # Pangkas huruf berulang ("baguuus" -> "bagus")
    text = re.sub(r'([a-zA-Z])\1+', r'\1', text)
    text = re.sub(r"@[A-Za-z0-9_]+", "", text)   # hapus mention
    text = re.sub(r"#[A-Za-z0-9_]+", "", text)   # hapus hashtag
    text = re.sub(r"\bRT\b", "", text)
    text = re.sub(r"http\S+", "", text)          # hapus URL
    text = re.sub(r"[0-9]+", "", text)           # hapus angka
    text = re.sub(r"[^\w\s]", "", text)          # hapus sisa karakter non-kata
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())                # rapikan spasi ganda
    return text.strip()


# Mapping emoji -> kata bermakna (emoji adalah sinyal sentimen yang kuat di review)
emoji_mapping = {
    "😂": "tertawa",
    "🤣": "tertawa",
    "😭": "sedih",
    "😢": "sedih",
    "😡": "marah",
    "🤬": "marah",
    "😠": "marah",
    "👍": "setuju",
    "👎": "tidak setuju",
    "🙏": "berharap",
    "❤️": "suka",
    "💙": "suka",
    "💔": "kecewa",
    "🤡": "badut",
    "💩": "buruk",
    "🔥": "menakjubkan",
    "⭐": "bintang",
    "🌟": "bintang",
    "😊": "senang",
    "😍": "senang",
    "😡": "marah",
    "🤮": "jijik",
    "😩": "lelah",
    "😑": "datar",
}


def convert_emoji(text):
    if not isinstance(text, str):
        return ""

    for emo, meaning in emoji_mapping.items():
        text = text.replace(emo, f" {meaning} ")
        text = text.replace("_", " ")

    # Emoji lain yang tidak ada di mapping didejomize menjadi teks (language=id)
    text = emoji.demojize(text, delimiters=("", ""), language="id")
    text = text.replace("_", " ")

    return " ".join(text.split())


def casefoldingText(text):
    return text.lower()


def fix_slangwords(text):
    words = text.split()
    fixed_words = [kamus_tidak_baku_dict.get(word, word) for word in words]
    return " ".join(fixed_words)


def filteringText(text):
    return stopword_remover.remove(text)


def stemmingText(text):
    return stemmer.stem(text)


def tokenizingText(text):
    return word_tokenize(text)

In [ ]:
# Kolom teks review dari Google Play Store adalah "content"
cleaning_df = pd.DataFrame({
    "raw_review": df["content"],
    "score": df["score"],  # rating bintang 1-5, disimpan untuk analisis perbandingan
})

cleaning_df.info()

### Menjalankan Pipeline Preprocessing

In [ ]:
# Langkah 0: konversi emoji menjadi kata
cleaning_df["text_demojize"] = cleaning_df["raw_review"].apply(convert_emoji)

# Langkah 1: pembersihan teks awal
cleaning_df["text_clean"] = cleaning_df["text_demojize"].apply(cleaningText)

# Langkah 2: huruf kecil semua (casefolding)
cleaning_df["text_casefolding"] = cleaning_df["text_clean"].apply(casefoldingText)

# Langkah 3: normalisasi kata tidak baku / singkatan
cleaning_df["text_slangwords"] = cleaning_df["text_casefolding"].apply(fix_slangwords)

# Langkah 4: hapus stopword (kata negasi tetap dipertahankan)
cleaning_df["text_stopword"] = cleaning_df["text_slangwords"].apply(filteringText)

# Langkah 5 (opsional): stemming — di-comment karena lambat untuk ~300 ribu baris
# cleaning_df["text_stemmed"] = cleaning_df["text_stopword"].apply(stemmingText)

# Langkah 6: tokenisasi (pecah menjadi list kata)
cleaning_df["text_token"] = cleaning_df["text_stopword"].apply(tokenizingText)

In [ ]:
cleaning_df.head()

## 4. Analisis Frekuensi Kata

Dihitung 2 versi frekuensi:

1. **Frekuensi awal** — dari teks review mentah, untuk melihat karakteristik bahasa review Play Store.
2. **Frekuensi akhir** — dari token hasil preprocessing, untuk melihat kata yang benar-benar
   masuk ke model klasifikasi.

In [ ]:
from collections import Counter

# Frekuensi dari teks review MENTAH (sebelum preprocessing)
raw_token_series = cleaning_df["raw_review"].apply(tokenizingText)

word_list = [token for row in raw_token_series for token in row]
word_freq = Counter(word_list)

sorted_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
df_freq = pd.DataFrame(sorted_freq, columns=["Kata", "Frekuensi"])

df_freq.to_csv("exports/frekuensi_awal_kata_shopee.csv", index=False)
print("Frekuensi kata awal tersimpan")
df_freq.head(20)

In [ ]:
# Frekuensi dari token HASIL PREPROCESSING
final_token_series = cleaning_df["text_token"].dropna()

word_list = [token for row in final_token_series for token in row]
word_freq = Counter(word_list)

sorted_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
df_freq = pd.DataFrame(sorted_freq, columns=["Kata", "Frekuensi"])

df_freq.to_csv("exports/frekuensi_akhir_kata_shopee.csv", index=False)
print("Frekuensi kata akhir tersimpan")
df_freq.head(20)

### Visualisasi Top 20 Kata Paling Sering Muncul

Grafik berikut menunjukkan 20 kata paling dominan setelah preprocessing — kata-kata ini
mencerminkan isu utama yang dibahas pengguna dalam review Shopee.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_n = 20
top_words = df_freq.head(top_n).sort_values(by="Frekuensi", ascending=True)

plt.figure(figsize=(12, 8))
sns.barplot(data=top_words, x="Frekuensi", y="Kata", color="steelblue")

plt.title("Top 20 Kata Paling Sering Muncul (Hasil Preprocessing)", fontsize=16)
plt.xlabel("Frekuensi")
plt.ylabel("Kata")
plt.tight_layout()

plt.savefig("shopee/images/top_20_frekuensi_kata.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df = pd.DataFrame({
    "raw_review": cleaning_df["raw_review"],
    "score": cleaning_df["score"],
    "clean_comment": cleaning_df["text_token"],
    "clean_text": cleaning_df["text_token"].apply(
        lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens)
    ),
})

df.head()

Review yang teksnya kosong setelah preprocessing (hanya emoji/spasi/stopword) dihapus di sini,
karena tidak memberi informasi apa pun bagi model.

In [ ]:
df["clean_text"] = df["clean_text"].fillna("")
df = df[df["clean_text"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)

print(f"Jumlah data setelah preprocessing: {df.shape[0]}")

df.to_csv("exports/cleaned_comments_shopee.csv", index=False)
print("Data hasil preprocessing tersimpan ke shopee/exports/cleaned_comments_shopee.csv")

## 5. Pelabelan Data (Lexicon-Based)

Karena review Play Store tidak memiliki label sentimen bawaan, label ditentukan secara **otomatis**
menggunakan **lexicon-based sentiment analysis**:

1. Setiap kata dicek ke dua kamus lexicon Indonesia ([lexicon_positive](https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv) & [lexicon_negative](https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv)).
2. Skor setiap kata dijumlahkan menjadi `polarity_score` per review.
3. Aturan penentuan polaritas:
   - Skor **> 0** → `positive`
   - Skor **< 0** → `negative`
   - Skor **= 0** → `neutral`

Ditambahkan juga lexicon custom khusus konteks e-commerce (kata-kata umum di review aplikasi belanja
yang tidak tercakup lexicon umum).

In [ ]:
df = pd.read_csv("exports/cleaned_comments_shopee.csv")

print(f"Jumlah data duplikat: {df.duplicated().sum()}")
print(f"Jumlah data kosong:\n{df.isna().sum()}")

In [ ]:
import ast

df["clean_comment"] = df["clean_comment"].apply(
    lambda x: ast.literal_eval(x)
    if isinstance(x, str) and x.startswith("[")
    else x
)

print(f"Tipe kolom clean_comment: {type(df['clean_comment'].iloc[0])}")

In [ ]:
# Ambil lexicon dari GitHub
import csv
import requests
from io import StringIO

lexicon_positive = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')

if response.status_code == 200:
    reader = csv.reader(StringIO(response.text), delimiter=',')
    for row in reader:
        lexicon_positive[row[0]] = int(row[1])
else:
    print("Failed to fetch positive lexicon data")

lexicon_negative = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')

if response.status_code == 200:
    reader = csv.reader(StringIO(response.text), delimiter=',')
    for row in reader:
        lexicon_negative[row[0]] = int(row[1])
else:
    print("Failed to fetch negative lexicon data")

print(f"Jumlah kata lexicon positive: {len(lexicon_positive)}")
print(f"Jumlah kata lexicon negative: {len(lexicon_negative)}")

### Lexicon Custom Khusus Konteks E-commerce

Kata-kata slang/khas review belanja online yang perlu masuk kamus agar polaritas akurat.

In [ ]:
# Tambahkan kata khas konteks review aplikasi belanja ke lexicon
custom_lexicon_negative = {
    "lemot": -2,
    "error": -1,
    "error2": -1,
    "scam": -3,
    "penipuan": -3,
    "tipu": -3,
    "zonk": -3,
    "jelek": -2,
    "rusak": -2,
    "cacat": -2,
    "susah": -1,
    "susah": -1,
    "lambat": -1,
    "kecewa": -2,
    "rugi": -2,
    "bohong": -2,
    "palsu": -2,
    "tidak sesuai": -2,
    "tidak bisa": -1,
    "error terus": -2,
    "tidak ada gunanya": -3,
    "parah": -2,
    "lemot": -2,
    "nyebelin": -2,
    "kesel": -2,
    "ribet": -1,
    "nyangkut": -1,
    "hilang": -1,
    "tidak sampai": -2,
    "tidak lengkap": -1,
}

custom_lexicon_positive = {
    "mantap": 2,
    "mantul": 3,
    "top": 2,
    "puas": 2,
    "cepat": 1,
    "ramah": 2,
    "mudah": 1,
    "praktis": 1,
    "murah": 1,
    "lengkap": 1,
    "keren": 2,
    "bagus2": 2,
    "memuaskan": 3,
    "recommended": 2,
    "rekomendasi": 1,
    "selamat": 1,
    "aman": 1,
    "helpful": 1,
    "responsif": 1,
    "fast respon": 2,
    "puas banget": 3,
}

for word, score in custom_lexicon_negative.items():
    lexicon_negative[word] = score

for word, score in custom_lexicon_positive.items():
    lexicon_positive[word] = score

print(f"Total lexicon positive (setelah custom): {len(lexicon_positive)}")
print(f"Total lexicon negative (setelah custom): {len(lexicon_negative)}")

### Fungsi Pelabelan Sentimen

In [ ]:
def sentiment_analysis_lexicon_indonesia(text):
    # Hitung skor polaritas sentimen sebuah review dari jumlah skor lexicon tiap kata
    score = 0

    for word in text:
        score = score + lexicon_positive.get(word, 0)
        score = score + lexicon_negative.get(word, 0)

    if score > 0:
        polarity = 'positive'
    elif score < 0:
        polarity = 'negative'
    else:
        polarity = 'neutral'

    return score, polarity

In [ ]:
results = list(df['clean_comment'].apply(sentiment_analysis_lexicon_indonesia))
results = list(zip(*results))
df['polarity_score'] = results[0]
df['polarity'] = results[1]

print("Distribusi label sentimen:")
print(df['polarity'].value_counts())

### Encode Label Menjadi Angka

- `positive` → **1**
- `neutral` → **0**
- `negative` → **2**

Pemetaan ini konsisten dengan konvensi yang dipakai saat visualisasi confusion matrix nanti.

In [ ]:
map_label = {
    'neutral': 0,
    'positive': 1,
    'negative': 2,
}

df['label'] = df['polarity'].map(map_label)

print(df['label'].value_counts().sort_index())
df.head()

In [ ]:
df.to_csv("shopee/exports/labeled_data.csv", index=False)
print("Data berlabel tersimpan ke exports/labeled_data.csv")

## 6. Analisis Distribusi Label Sentimen

Plot distribusi label untuk melihat **class imbalance**. Review Play Store umumnya didominasi
sentimen ekstrem (sangat puas / sangat kecewa), sehingga kelas `neutral` biasanya paling sedikit.
Kondisi ini yang menjadi alasan penggunaan `class_weight="balanced"` pada model dan pemilihan
metrik **macro F1** saat tuning.

In [ ]:
import matplotlib.pyplot as plt

polarity_counts = df['polarity'].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(
    polarity_counts.index,
    polarity_counts.values,
    color=['#2ECC71', "#8D8C88", "#9E352A"]
)

plt.title('Distribusi Sentimen Berdasarkan Label (Review Play Store Shopee)')
plt.xlabel('Label Sentimen')
plt.ylabel('Jumlah Data')

plt.tight_layout()
plt.savefig("images/distribusi_sentimen.png", dpi=300, bbox_inches="tight")
plt.show()

### Validasi Lexicon vs Rating Bintang

Kolom `score` (bintang 1–5) dari Play Store bisa dipakai sebagai *sanity check* hasil pelabelan lexicon:
review bintang 4–5 seharusnya didominasi label `positive`, dan bintang 1–2 didominasi `negative`.

In [ ]:
df["score_group"] = pd.cut(
    df["score"],
    bins=[0, 2, 3, 5],
    labels=["bintang 1-2", "bintang 3", "bintang 4-5"],
)

cross_tab = pd.crosstab(df["score_group"], df["polarity"], normalize="index")
print("Proporsi label lexicon per kelompok rating bintang:")
print(cross_tab.round(3))

cross_tab.plot(kind="bar", figsize=(8, 5), rot=0)
plt.title("Perbandingan Label Lexicon vs Rating Bintang")
plt.xlabel("Rating Bintang")
plt.ylabel("Proporsi")
plt.tight_layout()
plt.savefig("images/lexicon_vs_bintang.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Analisis Kata (Wordcloud)

Wordcloud memvisualisasikan kata-kata dominan; ukuran kata sebanding dengan frekuensinya.
Dibuat 3 versi: **unigram** (kata tunggal), **bigram** (2 kata), dan **trigram** (3 kata) —
bigram/trigram penting untuk menangkap frasa seperti "aplikasi error", "pengiriman cepat", atau
"tidak bisa login" yang bermakna utuh sebagai frasa.

In [ ]:
from collections import Counter
from collections.abc import Sequence
from typing import Any

import matplotlib.pyplot as plt
from wordcloud import WordCloud


def make_wordcloud(
    df: Any,
    token_column: str = "clean_comment",
    ngram_range: int | tuple[int, int] = 1,
    max_words: int = 200,
    width: int = 1200,
    height: int = 600,
    background_color: str = "white",
    colormap: str = "viridis",
    min_font_size: int = 10,
    random_state: int = 42,
    show: bool = True,
) -> tuple[WordCloud, Counter]:
    # Membuat wordcloud dari kolom yang berisi list token.
    # ngram_range: 1 = unigram, 2 = bigram, 3 = trigram, (1, 2) = unigram+bigram, dst.
    if isinstance(ngram_range, int):
        min_n, max_n = ngram_range, ngram_range
    else:
        min_n, max_n = ngram_range

    if min_n < 1 or max_n < min_n:
        raise ValueError("ngram_range harus berupa integer >= 1 atau tuple seperti (1, 2).")

    if token_column not in df.columns:
        raise KeyError(f"Kolom {token_column!r} tidak ditemukan.")

    token_counter: Counter = Counter()

    for tokens in df[token_column].dropna():
        if not isinstance(tokens, Sequence) or isinstance(tokens, (str, bytes)):
            continue

        normalized_tokens = [
            token.strip().lower()
            for token in tokens
            if isinstance(token, str) and token.strip()
        ]

        for n in range(min_n, max_n + 1):
            ngrams = (
                " ".join(normalized_tokens[i:i + n])
                for i in range(len(normalized_tokens) - n + 1)
            )
            token_counter.update(ngrams)

    if not token_counter:
        raise ValueError("Tidak ada token valid untuk dibuat wordcloud.")

    wordcloud = WordCloud(
        width=width,
        height=height,
        background_color=background_color,
        colormap=colormap,
        max_words=max_words,
        min_font_size=min_font_size,
        random_state=random_state,
        collocations=False,
    ).generate_from_frequencies(token_counter)

    if show:
        plt.figure(figsize=(width / 100, height / 100))
        plt.imshow(wordcloud, interpolation="bilinear")
        plt.axis("off")
        plt.show()

    return wordcloud, token_counter

In [ ]:
wc, frequencies = make_wordcloud(df, token_column="clean_comment", ngram_range=1)

In [ ]:
wc, frequencies = make_wordcloud(df, token_column="clean_comment", ngram_range=2)

In [ ]:
wc, frequencies = make_wordcloud(df, token_column="clean_comment", ngram_range=3)

## UnderSampling

In [1]:
import pandas as pd

labeled_data = pd.read_csv("exports/labeled_data.csv")

In [2]:
# Tampilkan jumlah data setiap label sebelum undersampling
print("Jumlah data sebelum undersampling:")
print(labeled_data["polarity"].value_counts())

# Pisahkan fitur dan label
X = labeled_data["clean_text"]
y = labeled_data["label"]

# Undersampling berdasarkan jumlah kelas paling sedikit
data_train = pd.DataFrame({
    "text": X,
    "label": y
})

jumlah_minimum = data_train["label"].value_counts().min()

data_undersampled = (
    data_train
    .groupby("label", group_keys=False)
    .sample(n=jumlah_minimum, random_state=42)
    .sample(frac=1, random_state=42)
)

# Tampilkan jumlah data setelah undersampling
print("\nJumlah data setelah undersampling:")
print(data_undersampled["label"].value_counts().sort_index())

# Gunakan data hasil undersampling
X = data_undersampled["text"]
y = data_undersampled["label"]

Jumlah data sebelum undersampling:
polarity
negative    507908
positive    378723
neutral     144484
Name: count, dtype: int64

Jumlah data setelah undersampling:
label
0    144484
1    144484
2    144484
Name: count, dtype: int64


In [3]:
data_undersampled.to_csv("exports/undersampled_data.csv")

## **SKEMA 1: SVM + TF-IDF (80/20)**

**SVM (Support Vector Machine)** mencari hyperplane pemisah kelas dengan margin maksimal —
sangat efektif untuk teks berdimensi tinggi seperti fitur TF-IDF. **TF-IDF** mengubah teks
menjadi vektor bobot kata: kata yang sering muncul di satu dokumen tapi jarang di dokumen lain
mendapat bobot tinggi.

- `ngram_range=(1, 2)` → unigram + bigram, menangkap frasa seperti "tidak bisa".
- `min_df=2` → abaikan kata yang muncul < 2 kali (mengurangi noise).
- `sublinear_tf=True` → redam pengaruh kata yang berulang banyak kali dalam satu review.
- `class_weight="balanced"` → kompensasi class imbalance.
- Data dibagi **80% train / 20% test** dengan `stratify=y` agar proporsi label sama di train dan test.

In [ ]:
import pandas as pd

df = pd.read_csv("exports/undersampled_data.csv")
df.head()

### Split Data

In [ ]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

### Train Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=5000,
        sublinear_tf=True,
    )),
    ("svm", LinearSVC(C=1.0, class_weight="balanced", random_state=42)),
])

In [ ]:
model_svm.fit(X_train, y_train)
print("Akurasi pada data uji:", accuracy_score(y_test, model_svm.predict(X_test)))

In [ ]:
# Evaluasi awal
pred_train = model_svm.predict(X_train)
pred_test = model_svm.predict(X_test)

print("Akurasi train:", accuracy_score(y_train, pred_train))
print("Akurasi test :", accuracy_score(y_test, pred_test))
print()
print(classification_report(y_test, pred_test))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels = ["positif", "netral", "negatif"]
cm = confusion_matrix(y_test, pred_test, labels=[1, 0, 2])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)

plt.title("Confusion Matrix - SVM + TF-IDF")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

In [ ]:
import joblib
joblib.dump(model_svm, "shopee/model/base_model_svm_tfidf.pkl")

### Tes model

In [ ]:
# sentimen = "ini aplikasi kok lelet banget di pakenya"
sentimen = "cepat bagus"

sentimen_bersih = tokenizingText(
    filteringText(
        fix_slangwords(
            casefoldingText(
                cleaningText(convert_emoji(sentimen))
            )
        )
    )
)
sentimen_bersih = " ".join(sentimen_bersih)

prediksi = model_svm.predict([sentimen_bersih])[0]
label_sentimen = {0: "neutral", 1: "positive", 2: "negative"}

print("Teks setelah preprocessing:", sentimen_bersih)
print("Prediksi sentimen:", label_sentimen[prediksi])

### Tuning Model (RandomizedSearchCV)

RandomizedSearchCV mencoba 50 kombinasi hyperparameter acak (dari ruang yang didefinisikan),
divalidasi dengan **Stratified K-Fold 5-fold** memakai metrik **macro F1** — macro F1 lebih adil
daripada akurasi untuk data imbalance karena menyeimbangkan kontribusi tiap kelas.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from scipy.stats import loguniform, uniform, randint

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("svm", LinearSVC(max_iter=5000))
])

param_distributions = {
    # Tuning TF-IDF
    "tfidf__analyzer": ["word", "char", "char_wb"],
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3), (2, 2)],
    "tfidf__min_df": randint(1, 10),
    "tfidf__max_df": uniform(0.5, 0.5),
    "tfidf__max_features": [10000, 20000, 22000, 25000, 50000, None],
    "tfidf__sublinear_tf": [True, False],
    "tfidf__norm": ["l1", "l2"],
    # Tuning SVM
    "svm__C": loguniform(1e-3, 1e2),
    "svm__class_weight": ["balanced"],
    "svm__loss": ["hinge", "squared_hinge"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV F1-macro:", random_search.best_score_)

best_model_svm = random_search.best_estimator_
y_pred_svm = best_model_svm.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("Akurasi pada data test:", accuracy_score(y_test, y_pred_svm))
print("Macro F1 pada data test:", f1_score(y_test, y_pred_svm, average="macro"))
print()
print(classification_report(y_test, y_pred_svm, digits=4))
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
import joblib
import os

os.makedirs("shopee/models", exist_ok=True)
joblib.dump(best_model_svm, "shopee/models/tuned_model_svm_tfidf.pkl")
print("Model SVM tersimpan ke shopee/models/tuned_model_svm_tfidf.pkl")

## **SKEMA 2: Random Forest + TF-IDF (70/30)**

**Random Forest** adalah ensamble ratusan decision tree yang voting mayoritas — lebih tahan
terhadap overfitting dibanding satu pohon dan tidak membutuhkan feature scaling.
Data dibagi **70% train / 30% test** untuk melihat pengaruh proporsi split yang berbeda
terhadap performa model.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

model_rf_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("rf", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

model_rf_tfidf.fit(X_train, y_train)
print("Akurasi pada data uji:", accuracy_score(y_test, model_rf_tfidf.predict(X_test)))

In [ ]:
import joblib
import os

os.makedirs("shopee/models", exist_ok=True)
joblib.dump(model_rf_tfidf, "shopee/models/base_model_rf_tfidf.pkl")
print("Model Random Forest tersimpan ke shopee/models/base_model_rf_tfidf.pkl")

### Tuning Random Forest (RandomizedSearchCV)

Ruang hyperparameter RF yang dituning: jumlah pohon (`n_estimators`), kedalaman pohon (`max_depth`),
minimum sampel untuk split/leaf, dan jumlah fitur per pohon (`max_features`). Karena melatih RF
dengan data besar + banyak pohon mahal, dipakai **3-fold CV** agar tuning selesai lebih cepat.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.stats import randint as sp_randint

param_distributions = {
    "tfidf__analyzer": ["word", "char", "char_wb"],
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3), (2, 2)],
    "tfidf__min_df": sp_randint(1, 10),
    "tfidf__max_features": [10000, 20000, 50000, None],
    "tfidf__sublinear_tf": [True, False],
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 20, 40, 60],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2"],
    "rf__class_weight": ["balanced"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model_rf_tfidf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
    return_train_score=True
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)
print("\nBest CV F1-macro:")
print(random_search.best_score_)

In [ ]:
best_model_rf = random_search.best_estimator_

y_pred_rf = best_model_rf.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Test Macro F1:", f1_score(y_test, y_pred_rf, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
import joblib
import os

os.makedirs("shopee/model", exist_ok=True)
joblib.dump(best_model_rf, "shopee/model/model_rf_tfidf.pkl")
print("Model Random Forest tersimpan ke shopee/model/model_rf_tfidf.pkl")

## **SKEMA 3: Logistic Regression + TF-IDF (80/20)**

**Logistic Regression** adalah baseline klasifikasi teks yang kuat, cepat, dan mudah diinterpretasi
(koefisien per fitur menunjukkan kontribusi tiap kata terhadap kelas). Dilakukan tuning dengan
**GridSearchCV** (mencoba semua kombinasi, bukan acak) pada grid yang relatif kecil.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

model_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=5000,
        sublinear_tf=True,
    )),
    ("lr", LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight="balanced"
    ))
])

model_lr.fit(X_train, y_train)
print("Akurasi pada data test:", accuracy_score(y_test, model_lr.predict(X_test)))

In [ ]:
import joblib
import os

os.makedirs("shopee/models", exist_ok=True)
joblib.dump(model_lr, "shopee/models/base_model_lr_tfidf.pkl")
print("Model Logistic Regression tersimpan ke shopee/models/base_model_lr_tfidf.pkl")

### Tuning Model (GridSearchCV)

GridSearchCV mengevaluasi **seluruh** kombinasi parameter pada grid (36 kombinasi di sini)
dengan 3-fold stratified CV dan metrik macro F1, lalu melatih ulang model terbaik (`refit=True`).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline_lr = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("lr", LogisticRegression(max_iter=2000, random_state=42))
])

param_grid_lr = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3],
    "tfidf__max_features": [5000, 10000],
    "lr__C": [0.1, 1, 10],
    "lr__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True
)

grid_lr.fit(X_train, y_train)

print("Best Parameters:")
print(grid_lr.best_params_)
print("Best CV Macro F1:", grid_lr.best_score_)

In [ ]:
best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Macro F1:", f1_score(y_test, y_pred_lr, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
import joblib
import os

os.makedirs("shopee/model", exist_ok=True)
joblib.dump(best_lr, "shopee/model/model_lr_tfidf.pkl")
print("Model Logistic Regression tersimpan ke shopee/model/model_lr_tfidf.pkl")

## **SKEMA 4: FastText Embedding + KNN (80/20)**

Berbeda dengan TF-IDF (matriks kata-agnostik), **FastText** melatih *word embedding* dari korpus
review itu sendiri: setiap kata dipetakan ke vektor dense, termasuk kata jarang/miskon (*out-of-vocabulary*)
karena FastText memakai subword (n-gram karakter). Representasi dokumen diperoleh dengan **rata-rata
vektor kata** (*average pooling*).

**KNN (K-Nearest Neighbors)** mengklasifikasikan dokumen baru berdasarkan mayoritas label dari
`k` dokumen terdekat di ruang embedding (menggunakan jarak cosine/euclidean).

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

### Custom Transformer FastText untuk Pipeline Scikit-Learn

Kelas `FastTextTransformer` membungkus gensim FastText menjadi estimator scikit-learn
(`fit`/`transform`) sehingga bisa dipakai langsung di dalam `Pipeline`.

In [ ]:
import numpy as np

from gensim.models import FastText
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class FastTextTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        vector_size=100,
        window=5,
        min_count=1,
        min_n=3,
        max_n=6,
        sg=0,
        epochs=20,
        workers=1,
        seed=42
    ):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.min_n = min_n
        self.max_n = max_n
        self.sg = sg
        self.epochs = epochs
        self.workers = workers
        self.seed = seed

    def _tokenize(self, text):
        if isinstance(text, str):
            return text.lower().split()
        return list(text)

    def fit(self, X, y=None):
        sentences = [self._tokenize(text) for text in X]

        self.fasttext_ = FastText(
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            min_n=self.min_n,
            max_n=self.max_n,
            sg=self.sg,
            workers=self.workers,
            seed=self.seed
        )

        self.fasttext_.build_vocab(sentences)

        self.fasttext_.train(
            sentences,
            total_examples=len(sentences),
            epochs=self.epochs
        )

        return self

    def transform(self, X):
        check_is_fitted(self, "fasttext_")

        document_embeddings = []

        for text in X:
            tokens = self._tokenize(text)

            word_vectors = [
                self.fasttext_.wv.get_vector(token)
                for token in tokens
            ]

            if word_vectors:
                document_vector = np.mean(word_vectors, axis=0)
            else:
                document_vector = np.zeros(self.vector_size, dtype=np.float32)

            document_embeddings.append(document_vector)

        return np.asarray(document_embeddings, dtype=np.float32)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

pipeline_fasttext_knn = Pipeline([
    ("fasttext", FastTextTransformer()),
    ("knn", KNeighborsClassifier())
])

pipeline_fasttext_knn.fit(X_train, y_train)

y_pred_ft_knn = pipeline_fasttext_knn.predict(X_test)

print("Akurasi:", accuracy_score(y_test, y_pred_ft_knn))
print("Macro F1:", f1_score(y_test, y_pred_ft_knn, average="macro"))
print()
print(classification_report(y_test, y_pred_ft_knn))

In [ ]:
import joblib
import os

os.makedirs("shopee/models", exist_ok=True)
joblib.dump(pipeline_fasttext_knn, "shopee/models/base_model_knn_fasttext.pkl")
print("Model KNN tersimpan ke shopee/models/base_model_knn_fasttext.pkl")

### Tuning FastText + KNN (RandomizedSearchCV)

Dituning parameter embedding FastText (dimensi vektor, jendela konteks, arah training CBOW/Skip-gram,
jumlah epoch) dan parameter KNN (`n_neighbors`, `weights`, `metric`). Mengingat tiap kombinasi harus
melatih FastText ulang dari nol, dipakai sampling 20 kombinasi dengan 3-fold CV.

In [ ]:
param_distributions = {
    "fasttext__vector_size": [50, 100, 200],
    "fasttext__window": [3, 5, 7],
    "fasttext__min_count": [1, 2],
    "fasttext__min_n": [2, 3],
    "fasttext__max_n": [5, 6],
    "fasttext__sg": [0, 1],
    "fasttext__epochs": [10, 20, 30],
    "knn__n_neighbors": [3, 5, 7, 9],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["cosine", "euclidean"]
}

param_distributions

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

random_search_ft = RandomizedSearchCV(
    estimator=pipeline_fasttext_knn,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True
)

random_search_ft.fit(X_train, y_train)

In [ ]:
print("Parameter terbaik:")
print(random_search_ft.best_params_)

print("\nSkor validasi terbaik (macro F1):")
print(random_search_ft.best_score_)

In [ ]:
best_model_ft_knn = random_search_ft.best_estimator_
y_pred_ft_knn = best_model_ft_knn.predict(X_test)

print("Akurasi test:", accuracy_score(y_test, y_pred_ft_knn))
print("Macro F1 test:", f1_score(y_test, y_pred_ft_knn, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ft_knn))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ft_knn))

In [ ]:
import joblib
import os

os.makedirs("shopee/model", exist_ok=True)
joblib.dump(best_model_ft_knn, "shopee/model/model_fasttext_knn.pkl")
print("Model FastText+KNN tersimpan ke shopee/model/model_fasttext_knn.pkl")

## **SKEMA 5: SentenceTransformer + Bi-LSTM (80/20)**

Skema deep learning. Model pre-trained multilingual
[`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
dipakai sebagai **feature extractor** (bobot dibekukan/`frozen`) untuk menghasilkan *token embeddings*,
kemudian di atasnya dilatih classifier **Bi-LSTM (Bidirectional LSTM)**:

- **LSTM** memproses urutan token dan menangkap konteks jangka panjang.
- **Bidirectional** → urutan dibaca dari kiri-kanan dan kanan-kiri sehingga tiap token memahami konteks dua arah.
- Representasi akhir diambil dari output LSTM pada posisi token terakhir (bukan padding),
  lalu melalui dropout dan lapisan linear ke 3 kelas.

Keuntungan: tidak perlu melatih embedding dari nol (knowledge bahasa dari pre-trained model),
sehingga lebih kuat pada teks pendek/informal seperti review aplikasi.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = [str(text) for text in X_train]
X_test = [str(text) for text in X_test]

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

print("Kelas:", label_encoder.classes_)
print("Jumlah kelas:", num_classes)
print("Train:", len(X_train))
print("Test:", len(X_test))

In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer

sentence_model_name = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

sentence_model = SentenceTransformer(sentence_model_name)

# Ambil tokenizer dan transformer encoder internal
tokenizer = sentence_model.tokenizer
transformer = sentence_model[0].auto_model

transformer.eval()

# Bekukan seluruh bobot transformer — hanya Bi-LSTM yang dilatih
for parameter in transformer.parameters():
    parameter.requires_grad = False

hidden_size = transformer.config.hidden_size

print("Hidden size:", hidden_size)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": self.labels[index]
        }

In [ ]:
batch_size = 16
max_length = 128

train_dataset = TextDataset(X_train, y_train_encoded, tokenizer, max_length=max_length)
test_dataset = TextDataset(X_test, y_test_encoded, tokenizer, max_length=max_length)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

In [ ]:
class SentenceTransformerBiLSTM(nn.Module):
    def __init__(
        self,
        transformer,
        hidden_size,
        num_classes,
        lstm_hidden_size=128,
        lstm_layers=1,
        dropout=0.3
    ):
        super().__init__()

        self.transformer = transformer

        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(dropout if lstm_layers > 1 else 0.0)
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(lstm_hidden_size * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            transformer_output = self.transformer(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        token_embeddings = transformer_output.last_hidden_state

        lstm_output, _ = self.bilstm(token_embeddings)

        # Ambil representasi token terakhir yang bukan padding
        lengths = attention_mask.sum(dim=1) - 1
        batch_indices = torch.arange(lstm_output.size(0), device=lstm_output.device)

        last_output = lstm_output[batch_indices, lengths]

        output = self.dropout(last_output)
        logits = self.classifier(output)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_bilstm = SentenceTransformerBiLSTM(
    transformer=transformer,
    hidden_size=hidden_size,
    num_classes=num_classes,
    lstm_hidden_size=128,
    lstm_layers=1,
    dropout=0.3
).to(device)

print("Device:", device)

### Training Bi-LSTM

- **AdamW** optimizer (Adam + weight decay untuk regularisasi).
- **CrossEntropyLoss** untuk klasifikasi 3 kelas.
- **Gradient clipping** (`max_norm=1.0`) mencegah gradient explosion khas RNN.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_bilstm.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

epochs = 10

for epoch in range(epochs):
    model_bilstm.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model_bilstm(input_ids=input_ids, attention_mask=attention_mask)

        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_bilstm.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        predictions = torch.argmax(logits, dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = total_loss / total_samples
    epoch_accuracy = total_correct / total_samples

    print(f"Epoch {epoch + 1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_accuracy:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_bilstm.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model_bilstm(input_ids=input_ids, attention_mask=attention_mask)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

all_labels = np.asarray(all_labels, dtype=int)
all_predictions = np.asarray(all_predictions, dtype=int)

class_names = [str(class_name) for class_name in label_encoder.classes_]
class_labels = np.arange(len(class_names))

print("Akurasi:", accuracy_score(all_labels, all_predictions))

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=class_labels,
        target_names=class_names,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions, labels=class_labels))

In [ ]:
import joblib
import os

os.makedirs("shopee/models", exist_ok=True)
joblib.dump(model_bilstm, "shopee/models/base_model_Bilstm_bert.pkl")
print("Model BiLSTM tersimpan ke shopee/models/base_model_Bilstm_bert.pkl")

### Tuning Bi-LSTM (Randomized Sampling + Validation Split)

Dipakai `ParameterSampler` untuk mencoba 10 kombinasi hyperparameter. Dataset train dibagi
menjadi **train subset (80%)** dan **validation subset (20%)**; pada tiap epoch model dinilai
di validation set dengan macro F1, dan bobot terbaik disimpan (*early stopping manual*).
Model terbaik lalu dievaluasi di test set.

In [ ]:
import copy
import random

from sklearn.model_selection import ParameterSampler
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, random_split

validation_ratio = 0.2
random_seed = 42

generator = torch.Generator().manual_seed(random_seed)

train_size = int((1 - validation_ratio) * len(train_dataset))
validation_size = len(train_dataset) - train_size

train_subset, validation_subset = random_split(
    train_dataset,
    [train_size, validation_size],
    generator=generator
)

print(f"Train subset: {len(train_subset)} | Validation subset: {len(validation_subset)}")

In [ ]:
parameter_distributions = {
    "batch_size": [8, 16, 32],
    "lstm_hidden_size": [64, 128, 256],
    "lstm_layers": [1, 2],
    "dropout": [0.2, 0.3, 0.5],
    "learning_rate": [1e-4, 3e-4, 1e-3],
    "weight_decay": [1e-5, 1e-4, 1e-3],
    "epochs": [5, 8, 10]
}

In [ ]:
def evaluate_model(model, data_loader, device):
    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)

            predictions = torch.argmax(logits, dim=1)

            all_predictions.extend(predictions.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    score = f1_score(all_labels, all_predictions, average="macro", zero_division=0)

    return score

In [ ]:
def train_trial(
    parameters,
    train_subset,
    validation_subset,
    transformer,
    hidden_size,
    num_classes,
    device
):
    batch_size = parameters["batch_size"]

    train_loader_trial = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    validation_loader_trial = DataLoader(
        validation_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    trial_model = SentenceTransformerBiLSTM(
        transformer=copy.deepcopy(transformer),
        hidden_size=hidden_size,
        num_classes=num_classes,
        lstm_hidden_size=parameters["lstm_hidden_size"],
        lstm_layers=parameters["lstm_layers"],
        dropout=parameters["dropout"]
    ).to(device)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        trial_model.parameters(),
        lr=parameters["learning_rate"],
        weight_decay=parameters["weight_decay"]
    )

    best_validation_f1 = -1.0
    best_state = None

    for epoch in range(parameters["epochs"]):
        trial_model.train()

        for batch in train_loader_trial:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            logits = trial_model(input_ids=input_ids, attention_mask=attention_mask)

            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(trial_model.parameters(), max_norm=1.0)

            optimizer.step()

        validation_f1 = evaluate_model(trial_model, validation_loader_trial, device)

        if validation_f1 > best_validation_f1:
            best_validation_f1 = validation_f1
            best_state = copy.deepcopy(trial_model.state_dict())

    trial_model.load_state_dict(best_state)

    return best_validation_f1, trial_model

In [ ]:
n_iter = 10

sampled_parameters = list(
    ParameterSampler(parameter_distributions, n_iter=n_iter, random_state=42)
)

tuning_results = []
best_score = -1.0
best_parameters = None
best_model_bilstm = None

for trial_number, parameters in enumerate(sampled_parameters, start=1):
    print(f"\nTrial {trial_number}/{n_iter}")
    print(parameters)

    validation_f1, trial_model = train_trial(
        parameters=parameters,
        train_subset=train_subset,
        validation_subset=validation_subset,
        transformer=transformer,
        hidden_size=hidden_size,
        num_classes=num_classes,
        device=device
    )

    result = {
        "trial": trial_number,
        "validation_f1_macro": validation_f1,
        **parameters
    }

    tuning_results.append(result)

    print(f"Validation F1-macro: {validation_f1:.4f}")

    if validation_f1 > best_score:
        best_score = validation_f1
        best_parameters = parameters
        best_model_bilstm = trial_model

print("\nParameter terbaik:")
print(best_parameters)

print("\nValidation F1-macro terbaik:")
print(best_score)

In [ ]:
import pandas as pd

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    by="validation_f1_macro",
    ascending=False
)

tuning_results_df

In [ ]:
best_model_bilstm.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = best_model_bilstm(input_ids=input_ids, attention_mask=attention_mask)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

all_labels = np.asarray(all_labels, dtype=int)
all_predictions = np.asarray(all_predictions, dtype=int)

class_names = [str(class_name) for class_name in label_encoder.classes_]
class_labels = np.arange(len(class_names))

print("Akurasi test:", accuracy_score(all_labels, all_predictions))
print("Macro F1 test:", f1_score(all_labels, all_predictions, average="macro"))

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=class_labels,
        target_names=class_names,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions, labels=class_labels))

In [ ]:
import os

os.makedirs("shopee/models", exist_ok=True)

torch.save(
    {
        "model_state_dict": best_model_bilstm.state_dict(),
        "best_parameters": best_parameters,
        "class_names": class_names,
        "hidden_size": hidden_size,
        "num_classes": num_classes,
        "max_length": max_length
    },
    "shopee/model/skema_5_sentence_transformer_bilstm.pt"
)

print("Model Bi-LSTM tersimpan ke shopee/model/skema_5_sentence_transformer_bilstm.pt")

## 8. Kesimpulan & Perbandingan Skema

Setiap skema selesai dievaluasi dengan **akurasi**, **macro F1**, classification report, dan
confusion matrix. Rekap hasil dapat dibuat dengan mengumpulkan skor tiap skema:

| Skema | Model | Rasio Split | Metrik Utama |
|---|---|---|---|
| 1 | SVM + TF-IDF (tuned) | 80/20 | Akurasi, Macro F1 |
| 2 | Random Forest + TF-IDF (tuned) | 70/30 | Akurasi, Macro F1 |
| 3 | Logistic Regression + TF-IDF (tuned) | 80/20 | Akurasi, Macro F1 |
| 4 | FastText + KNN (tuned) | 80/20 | Akurasi, Macro F1 |
| 5 | SentenceTransformer + Bi-LSTM (tuned) | 80/20 | Akurasi, Macro F1 |

**Catatan interpretasi:**

- Karena data imbalance, **macro F1** lebih representatif daripada akurasi dalam membandingkan skema.
- Model klasik (SVM/LR/RF) dengan TF-IDF sering menjadi *baseline* yang sulit dikalahkan untuk teks pendek seperti review.
- Skema 5 (deep learning) umumnya lebih kuat memahami konteks/frasa, tetapi jauh lebih mahal secara komputasi —
  gunakan GPU dan pertimbangkan pengurangan jumlah data/epoch jika waktu training terlalu lama.
- Label data adalah label lexicon (otomatis), sehingga performa model mempelajari *pola lexicon*,
  bukan kebenaran absolut sentimen manusia. Untuk hasil yang lebih kuat, validasi dengan anotasi manual pada sampel kecil.

## Implementasi

In [ ]:
import joblib

model_svm = joblib.load("shopee/models/tuned_model_svm_tfidf.pkl")
model_lr = joblib.load("shopee/models/base_model_lr_tfidf.pkl")

In [ ]:
test_cases = [
    # Kategori: Negatif
    "Aplikasi ini sering crash dan sangat lambat saat dibuka.",
    "Pelayanannya buruk sekali, saya kecewa dan tidak akan kembali lagi.",
    
    # Kategori: Netral
    "Aplikasi baru saja diperbarui ke versi terbaru.",
    "Saya membeli produk ini kemarin di toko dekat rumah.",
    
    # Kategori: Positif
    "Sangat puas dengan kualitas barangnya, pengiriman juga cepat",
    "Fitur terbarunya keren banget dan sangat membantu pekerjaan saya."
]
map_label = {
     0 : "neutral",
    1 : "positive",
    2 : "negative",
}

for text in test_cases:
    print("Teks:", text)
    print("Prediksi label:", map_label[model_lr.predict([text.lower()])[0]])
    
    
# kurang handle imbalance data, misal prediksi semua jadi negatif, padahal ada positif  dan netral. Bisa diatasi dengan oversampling atau undersampling, atau menggunakan class_weight di model.